# 面试问题：怎样用 Roofline 分析 LLM Prefill/Decode，并做容量规划？

**回答主线。** Roofline 用算术强度 `AI = FLOPs / bytes moved` 连接算法和硬件，性能上界为 `min(peak FLOP/s, bandwidth * AI)`。Prefill 的大矩阵乘通常能形成较高 AI，decode 在小 batch 下反复读权重与 KV，常受带宽约束。这个判断解释了为什么 batching、量化、融合和 P/D 资源配置有效，但不能替代真实 profiling。

下面从 GEMM 和线性层估算 FLOPs/bytes，计算 ridge point、延迟下界、batch 摊销、SLO 容量与模型—实测偏差。所有硬件数字都是受控假设，不对应某块 GPU 的承诺性能。


In [ ]:
import math  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 单位统一为 FLOP、byte、second，避免 TFLOPS/GB/s 混算千倍。
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Hardware155:  # 定义承载本节状态与行为的数据结构。
    name: str  # 执行当前语句以推进本节示例。
    peak_tflops: float  # 执行当前语句以推进本节示例。
    bandwidth_gbs: float  # 执行当前语句以推进本节示例。
    hbm_gb: float  # 执行当前语句以推进本节示例。

toy_gpu155 = Hardware155("toy-accelerator", 120.0, 2000.0, 80.0)  # 计算并保存当前步骤的中间状态。
assert toy_gpu155.peak_tflops > 0  # 用受控断言验证关键不变量。
assert toy_gpu155.bandwidth_gbs > 0  # 用受控断言验证关键不变量。
assert toy_gpu155.hbm_gb == 80.0  # 用受控断言验证关键不变量。


## 1. Ridge point 区分带宽区与计算区

当 `AI < peak/bandwidth`，移动数据更慢，处于斜线带宽区；超过 ridge 后受峰值计算限制。峰值只是理论 ceiling，还需乘以可达到的 kernel efficiency。


In [ ]:
def roofline155(hardware, arithmetic_intensity):  # 定义本节可复用的核心函数。
    # bandwidth*AI 得到 GFLOP/s，再除 1000 转为 TFLOP/s。
    bandwidth_bound = hardware.bandwidth_gbs * arithmetic_intensity / 1000.0  # 计算并保存当前步骤的中间状态。
    attainable = min(hardware.peak_tflops, bandwidth_bound)  # 计算并保存当前步骤的中间状态。
    ridge = hardware.peak_tflops * 1000.0 / hardware.bandwidth_gbs  # 计算并保存当前步骤的中间状态。
    regime = "memory" if arithmetic_intensity < ridge else "compute"  # 计算并保存当前步骤的中间状态。
    return attainable, ridge, regime  # 返回当前分支计算出的结果。

low155 = roofline155(toy_gpu155, 10.0)  # 计算并保存当前步骤的中间状态。
high155 = roofline155(toy_gpu155, 100.0)  # 计算并保存当前步骤的中间状态。
assert low155[2] == "memory"  # 用受控断言验证关键不变量。
assert high155[2] == "compute"  # 用受控断言验证关键不变量。
assert math.isclose(low155[1], 60.0)  # 用受控断言验证关键不变量。


## 2. GEMM 的 AI 取决于矩阵形状与数据复用

`M×K` 乘 `K×N` 约需 `2MKN` FLOPs。最低数据移动包含 A、B、C 各一次，但缓存失配会更多。Prefill 的 `M=batch*sequence` 大，能摊薄权重 B；decode 的 M 小，权重字节占主导。


In [ ]:
def gemm_work155(m, n, k, bytes_per_value=2):  # 定义本节可复用的核心函数。
    # 采用理想各矩阵只读写一次的下界，真实 kernel 只能更差。
    flops = 2.0 * m * n * k  # 计算并保存当前步骤的中间状态。
    moved = bytes_per_value * (m * k + k * n + m * n)  # 计算并保存当前步骤的中间状态。
    return {"flops": flops, "bytes": moved, "ai": flops / moved}  # 返回当前分支计算出的结果。

prefill155 = gemm_work155(m=2048, n=4096, k=4096)  # 计算并保存当前步骤的中间状态。
decode155 = gemm_work155(m=8, n=4096, k=4096)  # 计算并保存当前步骤的中间状态。
assert prefill155["ai"] > decode155["ai"]  # 用受控断言验证关键不变量。
assert prefill155["flops"] > decode155["flops"]  # 用受控断言验证关键不变量。
assert decode155["bytes"] > 0  # 用受控断言验证关键不变量。


## 3. LLM 线性层估算还要区分权重、激活和 KV

Decode 每个 token 至少读取主要权重；attention 还随上下文读取 KV。下面把二者分开，便于判断量化权重或量化 KV 各能优化哪部分，而不是把所有字节都算作参数。


In [ ]:
def decode_step_work155(parameter_count, batch, hidden, layers, context, kv_heads, head_dim, weight_bytes=2, kv_bytes=2):  # 定义本节可复用的核心函数。
    # 参数 GEMV 近似每个参数一次乘加；KV 为 K/V 两份并按 batch 读取。
    flops = 2.0 * parameter_count * batch  # 计算并保存当前步骤的中间状态。
    weight_traffic = parameter_count * weight_bytes  # 计算并保存当前步骤的中间状态。
    kv_traffic = 2.0 * layers * batch * context * kv_heads * head_dim * kv_bytes  # 计算并保存当前步骤的中间状态。
    activation_traffic = batch * hidden * layers * 4  # 计算并保存当前步骤的中间状态。
    total_bytes = weight_traffic + kv_traffic + activation_traffic  # 计算并保存当前步骤的中间状态。
    return {"flops": flops, "weight_bytes": weight_traffic, "kv_bytes": kv_traffic, "bytes": total_bytes, "ai": flops / total_bytes}  # 返回当前分支计算出的结果。

step155 = decode_step_work155(int(7e9), 8, 4096, 32, 4096, 8, 128)  # 计算并保存当前步骤的中间状态。
assert step155["weight_bytes"] > step155["kv_bytes"]  # 用受控断言验证关键不变量。
assert step155["bytes"] > step155["weight_bytes"]  # 用受控断言验证关键不变量。
assert step155["ai"] > 0  # 用受控断言验证关键不变量。


## 4. 延迟下界取计算时间与数据时间的最大值

`flops/peak` 和 `bytes/bandwidth` 可以重叠，因此理想下界取最大值。实测小于这个值说明单位或流量估算错误；实测更大则可能来自 kernel efficiency、同步、调度和通信。


In [ ]:
def latency_floor155(work, hardware):  # 定义本节可复用的核心函数。
    # peak_tflops 与 bandwidth_gbs 分别换算成每秒基础单位。
    compute_s = work["flops"] / (hardware.peak_tflops * 1e12)  # 计算并保存当前步骤的中间状态。
    memory_s = work["bytes"] / (hardware.bandwidth_gbs * 1e9)  # 计算并保存当前步骤的中间状态。
    return {"compute_s": compute_s, "memory_s": memory_s, "floor_s": max(compute_s, memory_s), "regime": "compute" if compute_s >= memory_s else "memory"}  # 返回当前分支计算出的结果。

floor_prefill155 = latency_floor155(prefill155, toy_gpu155)  # 计算并保存当前步骤的中间状态。
floor_decode155 = latency_floor155(step155, toy_gpu155)  # 计算并保存当前步骤的中间状态。
assert floor_prefill155["floor_s"] >= floor_prefill155["compute_s"]  # 用受控断言验证关键不变量。
assert floor_decode155["floor_s"] >= floor_decode155["memory_s"]  # 用受控断言验证关键不变量。
assert floor_decode155["regime"] == "memory"  # 用受控断言验证关键不变量。


## 5. Decode batching 用更多 FLOPs 换权重读取摊销

权重可在 batch 内复用，batch 增长时 FLOPs 线性增加而权重流量近似不变，AI 上升。但更大 batch 会增加 KV 容量、排队和 TPOT 抖动，所以吞吐最优不一定满足交互 SLO。


In [ ]:
batches155 = np.array([1, 2, 4, 8, 16, 32])  # 计算并保存当前步骤的中间状态。
works155 = [decode_step_work155(int(7e9), int(b), 4096, 32, 2048, 8, 128) for b in batches155]  # 计算并保存当前步骤的中间状态。
ais155 = np.array([w["ai"] for w in works155])  # 计算并保存当前步骤的中间状态。
floors155 = np.array([latency_floor155(w, toy_gpu155)["floor_s"] for w in works155])  # 计算并保存当前步骤的中间状态。
per_token155 = floors155 / batches155  # 计算并保存当前步骤的中间状态。
# 理想模型中 batch 增长摊薄固定权重读取，但总 step 时间不会下降为负。
assert np.all(np.diff(ais155) > 0)  # 用受控断言验证关键不变量。
assert per_token155[-1] < per_token155[0]  # 用受控断言验证关键不变量。
assert np.all(floors155 > 0)  # 用受控断言验证关键不变量。


## 6. SLO 容量规划从 service demand 与利用率开始

若每请求平均消耗 `prompt_tokens*prefill_ms + output_tokens*decode_ms` 的设备时间，单副本稳定容量受目标利用率约束。还要按长度分布而非平均值压测，因为长 prompt 会制造 head-of-line blocking。


In [ ]:
def replicas_needed155(arrival_rps, prompt_tokens, output_tokens, prefill_ms_token, decode_ms_token, target_utilization=0.7):  # 定义本节可复用的核心函数。
    # 把每请求两阶段设备占用相加，再按每秒可用毫秒计算副本数。
    demand_ms = prompt_tokens * prefill_ms_token + output_tokens * decode_ms_token  # 计算并保存当前步骤的中间状态。
    capacity_per_replica = 1000.0 * target_utilization / demand_ms  # 计算并保存当前步骤的中间状态。
    return math.ceil(arrival_rps / capacity_per_replica), demand_ms, capacity_per_replica  # 返回当前分支计算出的结果。

replicas155, demand155, capacity155 = replicas_needed155(20, 1000, 200, 0.002, 0.04)  # 计算并保存当前步骤的中间状态。
assert demand155 == 10.0  # 用受控断言验证关键不变量。
assert math.isclose(capacity155, 70.0)  # 用受控断言验证关键不变量。
assert replicas155 == 1  # 用受控断言验证关键不变量。


## 7. 用实测/上界比定位模型遗漏而非粉饰利用率

Roofline 上界乘以 kernel efficiency 后才接近可达性能。若实测远低于 ceiling，应分解启动、同步、通信、padding 和调度；不能把所有差距叫作“GPU 利用率低”。


In [ ]:
def efficiency_report155(work, hardware, measured_s):  # 定义本节可复用的核心函数。
    # measured_s 必须不小于物理下界；效率按下界/实测定义。
    floor = latency_floor155(work, hardware)["floor_s"]  # 计算并保存当前步骤的中间状态。
    if measured_s + 1e-12 < floor:  # 按当前条件选择后续控制路径。
        raise ValueError("measurement beats declared hardware bound; check units")  # 遇到非法合同立即显式失败。
    return {"floor_s": floor, "measured_s": measured_s, "efficiency": floor / measured_s, "overhead_s": measured_s - floor}  # 返回当前分支计算出的结果。

measured155 = floor_decode155["floor_s"] * 1.8  # 计算并保存当前步骤的中间状态。
report155 = efficiency_report155(step155, toy_gpu155, measured155)  # 计算并保存当前步骤的中间状态。
assert 0 < report155["efficiency"] < 1  # 用受控断言验证关键不变量。
assert report155["overhead_s"] > 0  # 用受控断言验证关键不变量。
assert math.isclose(report155["efficiency"], 1 / 1.8)  # 用受控断言验证关键不变量。


## 8. 硬件选择同时受 HBM 准入与 SLO 约束

算得快但放不下权重和目标 KV batch 的设备不是候选。下面先做内存准入，再用延迟下界排序；真实采购还要加入互联、可用 kernel、功耗、价格与故障域。


In [ ]:
def choose_hardware155(work, required_hbm_gb, candidates):  # 定义本节可复用的核心函数。
    # 先过滤容量，再按理论下界排序，并保留完整诊断结果。
    feasible = []  # 计算并保存当前步骤的中间状态。
    for hw in candidates:  # 遍历输入元素以累积或检查结果。
        if hw.hbm_gb >= required_hbm_gb:  # 按当前条件选择后续控制路径。
            feasible.append((latency_floor155(work, hw)["floor_s"], hw.name))  # 执行当前语句以推进本节示例。
    if not feasible:  # 按当前条件选择后续控制路径。
        return None  # 返回当前分支计算出的结果。
    return min(feasible)  # 返回当前分支计算出的结果。

candidates155 = [toy_gpu155, Hardware155("bandwidth-heavy", 80, 3000, 96), Hardware155("too-small", 200, 3000, 24)]  # 计算并保存当前步骤的中间状态。
choice155 = choose_hardware155(step155, required_hbm_gb=40, candidates=candidates155)  # 计算并保存当前步骤的中间状态。
assert choice155 is not None  # 用受控断言验证关键不变量。
assert choice155[1] == "bandwidth-heavy"  # 用受控断言验证关键不变量。
assert choose_hardware155(step155, 120, candidates155) is None  # 用受控断言验证关键不变量。


## 面试总结

- Roofline 上界是 `min(peak compute, bandwidth × arithmetic intensity)`，ridge point 决定优化方向。
- Prefill 通常因大 M 获得较高复用；小 batch decode 重复读权重/KV，更容易受带宽限制。
- Batch、量化和融合通过提高 AI 或减少字节改善上界，但可能增加排队、KV 容量和尾延迟。
- 容量规划必须把理论下界、实测效率、长度分布、HBM 准入与 SLO 一起版本化。

延伸阅读：[Berkeley Roofline Technical Report](https://www2.eecs.berkeley.edu/Pubs/TechRpts/2008/EECS-2008-134.pdf)、[LLM Inference Roofline Survey](https://arxiv.org/abs/2402.16363)、[DistServe](https://arxiv.org/abs/2401.09670)。
